# SSY340 Project: Can a DNN help a 3-Lead ECG?

## Data Preprossessing

In [3]:
import ast
from pathlib import Path
import numpy as np
import pandas as pd
import wfdb
import kagglehub

/Users/donky/Documents/SSY340/DML-project/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Pin the Kaggle version so everyone uses the same dataset.
# KaggleHub downloads once and reuses its local cache on later runs.
path = kagglehub.dataset_download("khyeh0719/ptb-xl-dataset/versions/1")
download_dir = Path(path)

# Locate the metadata even when the download contains a nested folder.
metadata_files = list(download_dir.rglob("ptbxl_database.csv"))
if len(metadata_files) != 1:
    raise FileNotFoundError(
        f"Expected one ptbxl_database.csv under {download_dir}, found {len(metadata_files)}"
    )
DATA_DIR = metadata_files[0].parent
print("Dataset folder:", DATA_DIR)

meta = pd.read_csv(DATA_DIR / "ptbxl_database.csv", index_col="ecg_id")
meta["scp_codes"] = meta["scp_codes"].apply(ast.literal_eval)

print(meta.shape)   # expect (21837, 27): one row per record
print(meta[["patient_id", "scp_codes", "strat_fold", "filename_lr"]].head())

In [ ]:
scp = pd.read_csv(DATA_DIR / "scp_statements.csv", index_col=0)
scp = scp[scp["diagnostic"] == 1]          # keep only diagnostic codes

def to_superclasses(codes):
    return sorted({scp.loc[c, "diagnostic_class"] for c in codes if c in scp.index})

meta["superclass"] = meta["scp_codes"].apply(to_superclasses)   # this line creates the column

print(meta["superclass"].head(10))
print(meta["superclass"].apply(len).value_counts())

In [ ]:
print(meta["superclass"].explode().value_counts())

In [ ]:
def load_signals(df, data_dir):
    X = np.empty((len(df), 1000, 12), dtype=np.float32)
    for i, f in enumerate(df["filename_lr"]):
        signal, _ = wfdb.rdsamp(str(data_dir / f))   # (1000, 12) in mV
        X[i] = signal
        if i % 2000 == 0:
            print(f"Loaded {i}/{len(df)}")
    return X

cache = DATA_DIR / "X100.npy"
if cache.exists():
    X = np.load(cache)
    print("Loaded from cache")
else:
    X = load_signals(meta, DATA_DIR)
    np.save(cache, X)
    print("Saved cache")

print(X.shape)   # expect (21837, 1000, 12)

### Removing data with no lables

In [ ]:
has_label = (meta["superclass"].apply(len) > 0).values

X = X[has_label]
meta = meta[has_label]

print(X.shape, meta.shape)   # expect (21430, 1000, 12) and (21430, 28)

In [ ]:
train_mask = (meta["strat_fold"] <= 8).values
val_mask   = (meta["strat_fold"] == 9).values
test_mask  = (meta["strat_fold"] == 10).values

X_train, X_val, X_test = X[train_mask], X[val_mask], X[test_mask]
y_train_raw = meta.loc[train_mask, "superclass"]
y_val_raw   = meta.loc[val_mask, "superclass"]
y_test_raw  = meta.loc[test_mask, "superclass"]

print(X_train.shape, X_val.shape, X_test.shape)

In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()
mlb.fit(y_train_raw)                       # learn the list of classes (from training data)
CLASS_NAMES = list(mlb.classes_)
print(CLASS_NAMES)                         # ['CD', 'HYP', 'MI', 'NORM', 'STTC'] (alphabetical)

y_train = mlb.transform(y_train_raw).astype("float32")
y_val   = mlb.transform(y_val_raw).astype("float32")
y_test  = mlb.transform(y_test_raw).astype("float32")

print(y_train.shape, y_val.shape, y_test.shape)   # (17111, 5) (2156, 5) (2163, 5)
print(y_train_raw.iloc[:3].tolist())
print(y_train[:3])

Removing bias and variance from training set

In [26]:
# Statistics per lead, computed from the TRAINING set only
mean = X_train.mean(axis=(0, 1), keepdims=True)   # shape (1, 1, 12)
std  = X_train.std(axis=(0, 1), keepdims=True)    # shape (1, 1, 12)
print("mean per lead:", mean.ravel().round(4))
print("std per lead: ", std.ravel().round(4))

X_train = (X_train - mean) / std
X_val   = (X_val   - mean) / std
X_test  = (X_test  - mean) / std

# Check: training data should now be ~0 mean, ~1 std for every lead
print(X_train.mean(axis=(0, 1)).round(3))
print(X_train.std(axis=(0, 1)).round(3))

mean per lead: [-0.0021 -0.0015  0.0005  0.0018 -0.0013 -0.0005  0.0002 -0.001  -0.0016
 -0.0014 -0.0008 -0.0024]
std per lead:  [0.1616 0.1553 0.162  0.1362 0.142  0.1365 0.2302 0.3287 0.3222 0.2863
 0.2661 0.2764]
[-0.  0. -0. -0.  0. -0.  0. -0.  0.  0. -0. -0.]
[1.002 1.002 1.001 1.002 1.001 1.002 1.001 1.    1.001 1.001 1.001 1.   ]


In [27]:
np.savez(DATA_DIR / "norm_stats.npz", mean=mean, std=std)

transpose for pytorch functions (N, 1000, 12) -> (N, 12, 1000)

In [28]:
X_train = np.ascontiguousarray(X_train.transpose(0, 2, 1))
X_val   = np.ascontiguousarray(X_val.transpose(0, 2, 1))
X_test  = np.ascontiguousarray(X_test.transpose(0, 2, 1))
print(X_train.shape, X_val.shape, X_test.shape)



(17111, 12, 1000) (2156, 12, 1000) (2163, 12, 1000)


In [ ]:
del X   # the full array is no longer needed

In [29]:
import torch
from torch.utils.data import Dataset, DataLoader

class ECGDataset(Dataset):
    def __init__(self, X, y):
        self.x = torch.from_numpy(X)   # (N, 12, 1000), float32
        self.y = torch.from_numpy(y)   # (N, 5), float32 multi-hot

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

train_ds = ECGDataset(X_train, y_train)
val_ds   = ECGDataset(X_val, y_val)
test_ds  = ECGDataset(X_test, y_test)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=256, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=256, shuffle=False)

x, y = next(iter(train_loader))
print(x.shape, x.dtype, y.shape, y.dtype)
# expect torch.Size([64, 12, 1000]) torch.float32 torch.Size([64, 5]) torch.float32

torch.Size([64, 12, 1000]) torch.float32 torch.Size([64, 5]) torch.float32
